# 03 · Prediction Intervals

A **$(1-\alpha)$ prediction interval** $[\hat\ell_{T+h},\hat u_{T+h}]$ is a range that should contain the *future
observation* with probability $1-\alpha$:
$$\Pr\left(\hat\ell_{T+h}\le Y_{T+h}\le\hat u_{T+h}\mid\mathcal I_T\right)=1-\alpha .$$
The usual **central** interval uses the $\alpha/2$ and $1-\alpha/2$ quantiles. A **one-sided** interval (e.g. "demand
will be below $\hat u$ with 95% probability") is what a service-level target needs.

## Theory

### Prediction interval ≠ confidence interval
| | Confidence interval | Prediction interval |
|---|---|---|
| covers | an unknown **parameter** / the conditional **mean** $\mathbb E[Y\mid x]$ | a future **observation** $Y$ |
| uncertainty included | estimation error only | estimation error **+ noise** |
| as $n\to\infty$ | shrinks to zero width | converges to the noise quantiles, never zero |

In regression: $\text{Var}(\hat y_0)=\sigma^2 x_0^\top(X^\top X)^{-1}x_0$ (CI) vs $\sigma^2\left(1+x_0^\top(X^\top X)^{-1}x_0\right)$ (PI).
Using a CI as a PI is one of the most common forecasting mistakes.

### Ways to build PIs
1. **Analytic**: $\hat y_{T+h}\pm z_{1-\alpha/2}\,\hat\sigma_h$ from the model (ARIMA/ETS formulas, nb 01).
2. **Simple-method formulas** (Hyndman & Athanasopoulos, FPP3 §5.5): naive $\sigma\sqrt h$, seasonal naive $\sigma\sqrt{k+1}$, $k=\lfloor (h-1)/m\rfloor$.
3. **Bootstrap**: simulate future paths by re-sampling past residuals; no normality assumption.
4. **Empirical (backtest) errors**: quantiles of *past out-of-sample* $h$-step errors; captures model misspecification.
5. **Conformal prediction**: has a coverage guarantee; nb 04.

### Why intervals are usually too narrow
Classical formulas ignore: (i) parameter-estimation uncertainty, (ii) model-selection uncertainty, (iii) future
structural change. Empirical studies (e.g. Makridakis et al.) show nominal 95% intervals often cover only 70–85%.
Backtest-based and conformal intervals fix (i)–(ii) automatically.

### Evaluating intervals
* **Coverage** (should equal $1-\alpha$). Not proper alone, because a huge interval always covers.
* **Width / sharpness** (smaller is better *given* coverage)
* **Interval (Winkler) score**, which is proper:
$$\text{IS}_\alpha(\ell,u;y)=(u-\ell)+\frac{2}{\alpha}(\ell-y)\mathbb 1\{y<\ell\}+\frac{2}{\alpha}(y-u)\mathbb 1\{y>u\}$$
* **Pointwise vs simultaneous**: an 80% band for each of 12 months does **not** contain the whole 12-month path 80% of the time.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(3)

y = pf.load_retail_total()
ly = np.log(y)

## 1. Confidence interval vs prediction interval
OLS on log-sales with trend + month dummies, trained on 2015–2022, forecasting 2023–2024.

In [2]:
tr, te = ly[:"2022-12-01"], ly["2023-01-01":]
Xtr, Xte = pf.seasonal_design(tr.index, "2015-01-01"), pf.seasonal_design(te.index, "2015-01-01")
ols = sm.OLS(tr, Xtr).fit()
sf = np.exp(ols.get_prediction(Xte).summary_frame(alpha=0.05))

fig = go.Figure()
fig.add_trace(go.Scatter(x=y.index[-48:], y=y[-48:], mode="lines", name="history / actual", line=dict(color=pf.HISTORY, width=1.5),
                         hovertemplate="actual %{y:,.0f}<extra></extra>"))
pf.add_band(fig, te.index, sf.obs_ci_lower, sf.obs_ci_upper, "95% PREDICTION interval (a new observation)", pf.SERIES[0], 0.22)
pf.add_band(fig, te.index, sf.mean_ci_lower, sf.mean_ci_upper, "95% CONFIDENCE interval (the mean)", pf.SERIES[1], 0.55)
fig.add_trace(go.Scatter(x=te.index, y=sf["mean"], mode="lines", name="point forecast", line=dict(color=pf.SERIES[0], width=2),
                         customdata=np.c_[sf.obs_ci_lower, sf.obs_ci_upper, sf.mean_ci_lower, sf.mean_ci_upper],
                         hovertemplate="forecast %{y:,.0f}<br>PI [%{customdata[0]:,.0f}, %{customdata[1]:,.0f}]"
                                       "<br>CI [%{customdata[2]:,.0f}, %{customdata[3]:,.0f}]<extra></extra>"))
fig.update_layout(title="A confidence interval for the mean is far narrower than a prediction interval",
                  hovermode="x unified", yaxis_title="sales")
fig.show()

yt = np.exp(te)
print(f"coverage of actuals: CI = {pf.coverage(yt, sf.mean_ci_lower, sf.mean_ci_upper):.2f}   PI = {pf.coverage(yt, sf.obs_ci_lower, sf.obs_ci_upper):.2f}")

coverage of actuals: CI = 0.46   PI = 1.00


## 2. Rolling-origin backtest of four interval methods

Monthly origins from month 48 onward; at every origin we refit and forecast $h=1..12$.

| Method | Description |
|---|---|
| **Analytic SARIMA** | airline model on log y, Gaussian $\pm z\sigma_h$ |
| **Bootstrap SARIMA** | same model, 1000 simulated paths with *re-sampled residuals* |
| **Empirical errors** | SARIMA point forecast + quantiles of **past out-of-sample** $h$-step log errors |
| **Seasonal naive** | last year's value + drift, Gaussian with the $\sigma\sqrt{k+1}$ formula |

The bootstrap is written out explicitly so the mechanics are visible. The airline model on $z=\log y$ is
$z_t = z_{t-1}+z_{t-12}-z_{t-13}+e_t+\theta e_{t-1}+\Theta e_{t-12}+\theta\Theta e_{t-13}$.
We iterate this recursion forward with future $e_t$ drawn from the empirical residuals.

In [3]:
def bootstrap_airline(z, res, h, n_paths, rng):
    th, Th = res.params["ma.L1"], res.params["ma.S.L12"]
    resid = np.asarray(res.resid)
    pool = resid[13:] - resid[13:].mean()           # drop start-up residuals, centre
    n = len(z)
    zz = np.tile(np.r_[z, np.zeros(h)], (n_paths, 1))
    ee = np.tile(np.r_[resid, np.zeros(h)], (n_paths, 1))
    ee[:, n:] = rng.choice(pool, size=(n_paths, h))
    for t in range(n, n + h):
        zz[:, t] = (zz[:, t-1] + zz[:, t-12] - zz[:, t-13]
                    + ee[:, t] + th * ee[:, t-1] + Th * ee[:, t-12] + th * Th * ee[:, t-13])
    return np.exp(zz[:, n:])                         # (n_paths, h) on the original scale

H, LEVELS = 12, [0.8, 0.95]
first_origin, eval_from = 48, 72                     # evaluate only once 2 years of backtest errors exist
rows, point_err, all_paths = [], {}, {}
for o in range(first_origin, len(y)):
    z = ly.iloc[:o].to_numpy()
    hmax = min(H, len(y) - o)
    res = SARIMAX(ly.iloc[:o], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    fc = res.get_forecast(hmax)
    m, s = np.asarray(fc.predicted_mean), np.sqrt(np.asarray(fc.var_pred_mean))
    actual = y.iloc[o:o + hmax].to_numpy()
    paths = bootstrap_airline(z, res, hmax, 1000, rng)
    all_paths[o] = paths
    # seasonal naive + drift on log scale
    d12 = z[12:] - z[:-12]
    sn_mean = np.array([z[o + h - 1 - 12] for h in range(1, hmax + 1)]) + d12.mean()
    sn_sd = d12.std(ddof=1) * np.sqrt(np.floor(np.arange(hmax) / 12) + 1)
    for h in range(1, hmax + 1):
        point_err[(o, h)] = np.log(actual[h - 1]) - m[h - 1]
        # empirical errors from origins whose h-step target was already observed at origin o
        past = [point_err[(j, h)] for j in range(first_origin, o - h + 1) if (j, h) in point_err]
        for lev in LEVELS:
            a = 1 - lev
            base = dict(origin=o, h=h, level=lev, y=actual[h - 1])
            zq = stats.norm.ppf(1 - a / 2)
            rows.append({**base, "method": "Analytic SARIMA", "lo": np.exp(m[h-1] - zq * s[h-1]), "hi": np.exp(m[h-1] + zq * s[h-1])})
            rows.append({**base, "method": "Bootstrap SARIMA", "lo": np.quantile(paths[:, h-1], a / 2), "hi": np.quantile(paths[:, h-1], 1 - a / 2)})
            rows.append({**base, "method": "Seasonal naive", "lo": np.exp(sn_mean[h-1] - zq * sn_sd[h-1]), "hi": np.exp(sn_mean[h-1] + zq * sn_sd[h-1])})
            if len(past) >= 12:
                rows.append({**base, "method": "Empirical errors", "lo": np.exp(m[h-1] + np.quantile(past, a / 2)),
                             "hi": np.exp(m[h-1] + np.quantile(past, 1 - a / 2))})

bt = pd.DataFrame(rows)
bt = bt[bt.origin >= eval_from].copy()
bt["covered"] = (bt.y >= bt.lo) & (bt.y <= bt.hi)
bt["width"] = bt.hi - bt.lo
bt["interval_score"] = pf.interval_score(bt.y, bt.lo, bt.hi, 1 - bt.level)
print(f"{bt.origin.nunique()} evaluation origins, {len(bt)} interval forecasts")

48 evaluation origins, 4080 interval forecasts


In [4]:
summary = bt.groupby(["level", "method"]).agg(coverage=("covered", "mean"), mean_width=("width", "mean"),
                                               interval_score=("interval_score", "mean")).round(3)
summary

coverage  mean_width  interval_score
level method                                                
0.80  Analytic SARIMA      0.869     483.181         558.855
      Bootstrap SARIMA     0.888     518.013         576.997
      Empirical errors     0.796     402.705         556.831
      Seasonal naive       0.827     416.054         555.943
0.95  Analytic SARIMA      0.967     742.039         803.620
      Bootstrap SARIMA     0.973     788.062         827.824
      Empirical errors     0.931     620.283         764.650
      Seasonal naive       0.906     637.836         746.473

In [5]:
METHODS = ["Analytic SARIMA", "Bootstrap SARIMA", "Empirical errors", "Seasonal naive"]
COLOR = dict(zip(METHODS, pf.SERIES))                 # fixed colour per method, in every chart below
fig = pf.subplots(1, 2, titles=[f"{int(l*100)}% interval: empirical coverage by horizon (dashed = nominal)" for l in LEVELS],
                  shared_yaxes=True, height=420)
for c, lev in enumerate(LEVELS, start=1):
    g = bt[bt.level == lev].groupby(["method", "h"]).covered.mean().unstack(0)[METHODS]
    pf.add_lines_by(fig, g, COLOR, row=1, col=c, showlegend=c == 1, fmt=".2f")
    fig.add_hline(y=lev, line=dict(color=pf.INK["primary"], dash="dash", width=1), row=1, col=c)
fig.update_yaxes(range=[0.3, 1.02]); fig.update_xaxes(title_text="h (months)")
fig.update_layout(hovermode="x unified")
fig.show()

g = bt[bt.level == 0.8].groupby(["method", "h"]).width.mean().unstack(0)[METHODS]
fig = pf.add_lines_by(go.Figure(), g, COLOR, fmt=",.0f")
fig.update_layout(title="Mean 80% interval width by horizon", height=380, hovermode="x unified",
                  xaxis_title="h (months)", yaxis_title="width (sales units)")
fig.show()

**What to look for**
* Interval width should **grow with $h$** for every method except a pure-formula seasonal naive within one season.
* **Analytic and bootstrap SARIMA over-cover** (e.g. ~87% for a nominal 80%), so they are wider than necessary.
  The airline model differences the data twice. That treats every shock as permanent, while this series is closer to a
  trend + seasonality with *temporary* AR(1) shocks. So the model overstates how fast uncertainty grows: a
  **misspecified variance**, not a bug. Over-coverage is not "safe". The interval score's width term penalises it.
* **Empirical-error** intervals are built from what actually happened out-of-sample, so they absorb this
  misspecification. They land close to nominal at 80% and have competitive interval scores. At 95% they fall slightly
  short: with only 12–36 past errors per horizon the extreme quantiles are under-estimated. That is a classic
  small-sample trap, and conformal prediction (nb 04) fixes it with its $\lceil (n+1)(1-\alpha)\rceil$ correction.
* **Seasonal naive** is very competitive here because the seasonal pattern is stable. Always include a simple
  benchmark. Sophisticated models do not automatically produce better intervals.
* The interval score combines both: a method that is narrow *and* covers wins.

## 3. Pointwise vs simultaneous (path) coverage
If each month has an 80% interval, how often are **all 12 months** inside their intervals at once?

In [6]:
full = [o for o in all_paths if all_paths[o].shape[1] == H and o >= eval_from]
inside_all = []
for o in full:
    sub = bt[(bt.origin == o) & (bt.method == "Bootstrap SARIMA") & (bt.level == 0.8)]
    inside_all.append(sub.covered.all())
p = all_paths[full[-1]]
lo, hi = np.quantile(p, 0.1, axis=0), np.quantile(p, 0.9, axis=0)
sim_inside = np.mean(np.all((p >= lo) & (p <= hi), axis=1))
print(f"pointwise nominal coverage: 80%")
print(f"share of simulated PATHS entirely inside the pointwise band : {sim_inside:.2f}")
print(f"share of backtest origins whose 12 actuals were ALL inside   : {np.mean(inside_all):.2f}  ({len(full)} origins)")

pointwise nominal coverage: 80%
share of simulated PATHS entirely inside the pointwise band : 0.29
share of backtest origins whose 12 actuals were ALL inside   : 0.35  (37 origins)


A 12-month path stays entirely within pointwise 80% bands far less than 80% of the time, because each month gets a
fresh chance to escape. For questions about the whole path ("will we ever exceed capacity this year?") you need
**sample paths** (nb 05) or simultaneous bands (e.g. widen until 80% of *paths* are inside).

## 4. Practical guidance
* State the **nominal level**, check **empirical coverage per horizon** in a rolling backtest, and report the
  interval score.
* Prefer bootstrap / empirical / conformal intervals when residuals are non-Gaussian or the model is misspecified.
* Intervals on a transformed scale back-transform cleanly (they are quantiles), but they become asymmetric.
* For decisions with asymmetric costs, use a **one-sided** bound / a single quantile (nb 02, 12).

### Exercises
1. Add a "bootstrap + parameter uncertainty" variant: draw parameters from `res.cov_params()` before simulating.
2. Compute simultaneous 80% bands by scaling the pointwise band until 80% of bootstrap paths fit inside.
3. Replace Gaussian residuals in the analytic method with Student-t residuals.